# Example: Text Analysis with kdl-ai-coding-tools

This notebook demonstrates how to use the text analysis pipeline on a sample dataset.
It reads a text file, cleans the text, detects bigrams/trigrams, lemmatizes tokens,
and generates a word frequency bar chart.

In [ ]:
# Import required packages
import os
from nltk.corpus import stopwords
import nltk
from collections import defaultdict
import pandas as pd
from gensim.models.phrases import Phrases, Phraser
from gensim.utils import simple_preprocess
import spacy
import operator
import plotly as py
import plotly.express as px
import math

nltk.download('stopwords', quiet=True)

In [ ]:
# Set up paths
home_path = os.path.abspath(os.getcwd())
data_path = os.path.join(home_path, 'data')
results_path = os.path.join(home_path, 'results')
os.makedirs(results_path, exist_ok=True)

In [ ]:
# Load stop words
stop_words = []
stop_words.extend(stopwords.words('english'))
stop_words.extend(['would', 'said', 'says', 'also', 'good', 'lord', 'come', 'let', 'say', 'speak', 'know', 'hamlet'])

In [ ]:
# Define trigram builder
def make_trigrams(tokens, min_count=5, threshold=100):
    bigram = Phrases(tokens, min_count=min_count, threshold=threshold)
    bigram_model = Phraser(bigram)
    trigram = Phrases(bigram[tokens], threshold=threshold)
    trigram_model = Phraser(trigram)
    result = []
    for doc in tokens:
        bigram_result = bigram_model[doc]
        trigram_result = trigram_model[bigram_result]
        result.append(trigram_result)
    return result

In [ ]:
# Define lemmatization pipeline
def make_lemma(tokens):
    data_ngrams = make_trigrams(tokens)
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    texts_out = []
    for sent in data_ngrams:
        doc = nlp(' '.join(sent))
        texts_out.append([token.lemma_ for token in doc if token.lemma_ != '-PRON-'])
    return texts_out

In [ ]:
# Run the analysis on the test data file
filepath = os.path.join(data_path, 'test_data.txt')

# Read and tokenize the text
docs = []
with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        line = line.strip()
        if len(line) == 0:
            continue
        docs.append(line.split())

# Remove punctuation
words = []
for sentence in docs:
    words.append(simple_preprocess(str(sentence), deacc=True))

# Remove stop words
cleaned = []
for doc in words:
    processed = []
    for word in simple_preprocess(str(doc)):
        if word not in stop_words:
            processed.append(word)
    cleaned.append(processed)

# Lemmatize
lemma = make_lemma(cleaned)

# Count word frequencies
tokens = []
for sublist in lemma:
    for item in sublist:
        tokens.append(item)

freq = defaultdict(int)
for t in tokens:
    freq[t] += 1

freq = sorted(freq.items(), key=operator.itemgetter(1), reverse=True)
print(f'Top 10 words in {filepath}:')
for word, count in freq[:10]:
    print(f'  {word}: {count}')

In [ ]:
# Plot the top 10 words as a bar chart
top_n = 10
df = pd.DataFrame(freq[:top_n], columns=['Word', 'Count'])
df['Pct'] = ((df['Count'] / df['Count'].sum()) * 100).round(3).astype(str) + '%'

fig = px.bar(df, x='Word', y='Count', hover_data=[df['Pct']],
             text='Count', title=f'Top {top_n} Words',
             color='Word', color_discrete_sequence=px.colors.qualitative.Dark24)
fig.update_layout(
    title={'y': 0.90, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
    width=750, height=550, showlegend=False)
fig.update_xaxes(tickangle=-45)

# Save the figure
output_path = os.path.join(results_path, 'example_output.html')
py.offline.plot(fig, filename=output_path, auto_open=False)
print(f'Chart saved to {output_path}')
fig.show()

## Summary

This example ran the text analysis pipeline on `test_data.txt`.

- **Input**: Plain text file in `data/`
- **Output**: Interactive bar chart saved as HTML in `results/`
- **Parameters**: Stop word language, minimum n-gram count, n-gram threshold, number of top words to display

To analyze a different text file, change the `filepath` variable in the analysis cell above.